In [ ]:
# %%
import math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("results/figs", exist_ok=True)
plt.rcParams["figure.figsize"] = (9, 5)
QUICK = False
N_SEEDS = 40 if QUICK else 200

# ---- same world as the notebook (do not change, or results will not match) ----
_layout = np.random.default_rng(7)
N_NODES = 12
XY = _layout.uniform(0, 900, size=(N_NODES, 2))
SPEED = 130.0
T = np.ceil(np.linalg.norm(XY[:, None] - XY[None], axis=2) / SPEED).astype(int)
N_BINS = N_NODES - 1
BASE_RATE = _layout.uniform(20, 48, size=N_BINS)
DAY_MIN, START_HOUR = 720, 8


def demand_curve(h):
    return (0.25 + 1.2 * math.exp(-((h - 12.5) / 1.2) ** 2)
            + 0.6 * math.exp(-((h - 10.0) / 0.6) ** 2)
            + 0.6 * math.exp(-((h - 15.0) / 0.7) ** 2))


CURVE = np.array([demand_curve(START_HOUR + t / 60) for t in range(DAY_MIN)])


def fill_paths(seed, scale=1.0):
    """Fill level (%) of every bin every minute with NO robots: shape (DAY_MIN, N_BINS).
    Uses the same RNG draw order as simulate() so seed k is the same day."""
    rng = np.random.default_rng(seed)
    day_mult = rng.lognormal(0, 0.2, N_BINS)
    noise = np.clip(1 + 0.3 * rng.standard_normal((DAY_MIN, N_BINS)), 0, None)
    start = rng.uniform(0, 50, N_BINS)
    inc = BASE_RATE * day_mult * CURVE[:, None] * scale * noise / 60
    return np.minimum(start + np.cumsum(inc, axis=0), 100.0), day_mult


PATHS = [fill_paths(s)[0] for s in range(N_SEEDS)]
DAYMULT = np.array([fill_paths(s)[1] for s in range(N_SEEDS)])
print(f"{N_SEEDS} simulated days, {N_BINS} bins")